### Prepare 

In [39]:
from functions import *
print(check_jax_gpu())

Available GPUs:
Tesla V100-PCIE-32GB1: gpu
None


In [3]:
# necessary file path
settings_path = './settings_target/FZD7.json'
advanced_path = './settings_advanced/default_4stage_multimer.json'
filters_path = './settings_filters/default_filters.json'

# load settings json file
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

# get settings file name
settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

print("Target settings:")
print(json.dumps(target_settings, indent=4))
print('Filter Used:', filters_file)
print('Advanced Used:', advanced_file)

Target settings:
{
    "design_path": "/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7",
    "binder_name": "FZD7",
    "starting_pdb": "/hpf/projects/mtyers/ningrui/NXBindCraft/Targets/FZD7.pdb",
    "chains": "A",
    "target_hotspot_residues": "",
    "lengths": [
        50,
        100
    ],
    "number_of_final_designs": 20
}
Filter Used: default_filters
Advanced Used: default_4stage_multimer


In [4]:
# AF2 model settings (which model to use)
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

# set function paths (package settings)
bindcraft_folder = os.path.dirname('/hpf/projects/mtyers/ningrui/NXBindCraft/bindcraft.py') # main folder name, aka BindCraft...
advanced_settings["af_params_dir"] = bindcraft_folder
advanced_settings["dssp_path"] = os.path.join(bindcraft_folder, 'functions/dssp') # Define Secondary Structure of Proteins
advanced_settings["dalphaball_path"] = os.path.join(bindcraft_folder, 'functions/DAlphaBall.gcc') # computing solvent-accessible surface area (SASA) and buried surface area (BSA) in protein structures

# generate dicretories to store designs, stats and other output
design_paths = generate_directories(target_settings["design_path"])

# generate dataframes (possibility to store design stats)
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

# create csv file (path) to store stats 
trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

# create csv file with label
create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, filters_path)


In [5]:
# initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

# initialise counters
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0
rejected_designs = 0

┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2025.06+release.029c6a159b896477003a14f78f472d4cd2cead46 2025-02-04T15:14:13] retrieved from: http://www.pyrosetta.org


### Start design loop!!!

In [6]:
#### check if we have the target number of binders
#final_designs_reached = check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels)
#
#if final_designs_reached: +++++++++++++++++++
#    # stop design loop execution
#    break

# random seed to vary different design
seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])

# randomly find a binder length in the specified range [x,y]
samples = np.arange(min(target_settings["lengths"]), max(target_settings["lengths"]) + 1)
length = np.random.choice(samples)

# load desired helicity value --> dample different secondary structure contents
helicity_value = load_helicity(advanced_settings)

# generate design name & check if tranjectory was already run
design_name = target_settings["binder_name"] + "_l" + str(length) + "_s"+ str(seed) # l = length, s = seed
trajectory_dirs = ["Trajectory", "Trajectory/Relaxed", "Trajectory/LowConfidence", "Trajectory/Clashing"]
trajectory_exists = any(os.path.exists(os.path.join(design_paths[trajectory_dir], design_name + ".pdb")) for trajectory_dir in trajectory_dirs)


In [ ]:
# if not trajectory_exists: +++++++++++++++++++
print("Starting trajectory: "+design_name)

### begin binder hallucination 
# design_name: FZD7_l88_s105508
# length: 88
# seed: 105508
# helicity_value: -0.3
# design_models: [0, 1, 2, 3, 4]
# advanced_settings: <class 'dict'>
# design_paths: <class 'dict'>
# failure_csv: /hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/failure_csv.csv
trajectory = binder_hallucination(design_name, target_settings["starting_pdb"], target_settings["chains"],
                                            target_settings["target_hotspot_residues"], length, seed, helicity_value,
                                            design_models, advanced_settings, design_paths, failure_csv)
'''
### within binder_hallucination ###
design_logits --> stage 1 logit optimization (mixed of raw logits and softmax)
design_soft --> stage 1 logit optimization (only softmax)
'''


Starting trajectory: FZD7_l84_s145922
Stage 1: Test Logits
1 models [2] recycles 1 hard 0 soft 0.02 temp 1 loss 16.21 helix 1.77 pae 0.85 i_pae 0.89 con 5.09 i_con 4.58 plddt 0.28 ptm 0.55 i_ptm 0.07 rg 21.76
2 models [4] recycles 1 hard 0 soft 0.04 temp 1 loss 11.52 helix 1.17 pae 0.78 i_pae 0.82 con 4.97 i_con 4.43 plddt 0.33 ptm 0.56 i_ptm 0.10 rg 6.54
3 models [3] recycles 1 hard 0 soft 0.05 temp 1 loss 12.12 helix 1.46 pae 0.81 i_pae 0.86 con 4.61 i_con 4.56 plddt 0.31 ptm 0.57 i_ptm 0.09 rg 9.56
4 models [3] recycles 1 hard 0 soft 0.07 temp 1 loss 9.58 helix 1.00 pae 0.68 i_pae 0.74 con 4.18 i_con 4.40 plddt 0.39 ptm 0.59 i_ptm 0.16 rg 2.85
5 models [4] recycles 1 hard 0 soft 0.09 temp 1 loss 9.26 helix 0.73 pae 0.69 i_pae 0.81 con 3.90 i_con 4.52 plddt 0.41 ptm 0.55 i_ptm 0.10 rg 2.04
6 models [1] recycles 1 hard 0 soft 0.11 temp 1 loss 8.66 helix 0.88 pae 0.59 i_pae 0.69 con 3.69 i_con 4.20 plddt 0.49 ptm 0.59 i_ptm 0.18 rg 2.10
7 models [0] recycles 1 hard 0 soft 0.13 temp 1 l

In [17]:
af_model = mk_afdesign_model(protocol="binder", debug=False, data_dir=advanced_settings["af_params_dir"], 
                                use_multimer=advanced_settings["use_multimer_design"], num_recycles=advanced_settings["num_recycles_design"],
                                best_metric='loss')

In [ ]:
af_model.restart

In [ ]:
print(_af_prep.__dir__())

dict_keys(['protocol', '_num', '_args', 'opt', '_params', '_inputs', '_tmp', '_callbacks', '_cfg', '_model_params', '_model_names', 'prep_inputs', '_get_loss'])


In [ ]:
# difference between trajectory_interface_AA vs. trajectory_interface_residues ??
trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(trajectory_relaxed, binder_chain)

In [ ]:
mpnn_trajectories = mpnn_gen_sequence(trajectory_pdb, binder_chain, trajectory_interface_residues, advanced_settings)
existing_mpnn_sequences = set(pd.read_csv(mpnn_csv, usecols=['Sequence'])['Sequence'].values)

In [ ]:
mpnn_sequences = sorted({
                    mpnn_trajectories['seq'][n][-length:]: {
                        'seq': mpnn_trajectories['seq'][n][-length:],
                        'score': mpnn_trajectories['score'][n],
                        'seqid': mpnn_trajectories['seqid'][n]
                    } for n in range(advanced_settings["num_seqs"])
                    if (not restricted_AAs or not any(aa in mpnn_trajectories['seq'][n][-length:].upper() for aa in restricted_AAs))
                    and mpnn_trajectories['seq'][n][-length:] not in existing_mpnn_sequences
                }.values(), key=lambda x: x['score'])

### ______________________ Draft ______________________

In [13]:
# initialise binder hallucination model
af_model = mk_afdesign_model(protocol="binder", debug=False, data_dir=advanced_settings["af_params_dir"],
                             use_multimer=advanced_settings["use_multimer_design"], 
                             num_recycles=advanced_settings["num_recycles_design"],best_metric='loss')

In [23]:
print(af_model._args)
print(af_model._tmp)
print(af_model.opt)
print(af_model._params)
print(af_model._inputs)

{'use_templates': True, 'use_multimer': True, 'use_bfloat16': True, 'recycle_mode': 'last', 'use_mlm': False, 'realign': True, 'debug': False, 'repeat': False, 'homooligomer': False, 'copies': 1, 'optimizer': 'sgd', 'best_metric': 'loss', 'traj_iter': 1, 'traj_max': 10000, 'clear_prev': True, 'use_dgram': False, 'shuffle_first': True, 'use_remat': True, 'alphabet_size': 20, 'use_initial_guess': False, 'use_initial_atom_pos': False}
{'traj': {'seq': [], 'xyz': [], 'plddt': [], 'pae': []}, 'log': [], 'best': {}}
{'dropout': True, 'pssm_hard': True, 'learning_rate': 0.1, 'norm_seq_grad': True, 'num_recycles': 1, 'num_models': 1, 'sample_models': True, 'temp': 1.0, 'soft': 0.0, 'hard': 0.0, 'alpha': 2.0, 'con': {'num': 2, 'cutoff': 14.0, 'binary': False, 'seqsep': 9, 'num_pos': inf}, 'i_con': {'num': 1, 'cutoff': 21.6875, 'binary': False, 'num_pos': inf}, 'template': {'rm_ic': False}, 'weights': {'seq_ent': 0.0, 'plddt': 0.0, 'pae': 0.0, 'exp_res': 0.0, 'helix': 0.0}, 'fape_cutoff': 10.0}


In [27]:
target_hotspot_residues = None
af_model.prep_inputs(pdb_filename=target_settings["starting_pdb"], chain=target_settings["chains"], binder_len=length,
                    hotspot=target_hotspot_residues, seed=seed, rm_aa=advanced_settings["omit_AAs"],
                    rm_target_seq=advanced_settings["rm_template_seq_design"], 
                    rm_target_sc=advanced_settings["rm_template_sc_design"])

In [35]:
print(af_model._params['seq'].shape)
print(af_model._tmp)

(1, 95, 20)
{'traj': {'seq': [], 'xyz': [], 'plddt': [], 'pae': []}, 'log': [], 'best': {}}


In [19]:
af_model.__dir__()

['protocol',
 '_num',
 '_args',
 'opt',
 '_params',
 '_inputs',
 '_tmp',
 '_callbacks',
 '_cfg',
 '_model_params',
 '_model_names',
 'prep_inputs',
 '_get_loss',
 '__module__',
 '__init__',
 '_get_model',
 '__doc__',
 'set_weights',
 'set_seq',
 '_norm_seq_grad',
 'set_optimizer',
 'set_seed',
 'get_seq',
 'get_seqs',
 'rewire',
 '__dict__',
 '__weakref__',
 '__new__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__',
 '_get_seq',
 '_fix_pos',
 '_update_template',
 '_loss_fixbb',
 '_loss_binder',
 '_loss_partial',
 '_loss_hallucination',
 '_loss_unsupervised',
 '_prep_model',
 '_prep_features',
 '_prep_fixbb',
 '_prep_hallucination',
 '_prep_binder',
 '_prep_partial',
 'restart',
 '_get_model_nums',
 'run',
 '_single',
 '_recycle',
 'step',
 '_print_l

In [36]:
### Update weights based on specified settings
af_model.opt["weights"].update({"pae":advanced_settings["weights_pae_intra"],
                                "plddt":advanced_settings["weights_plddt"],
                                "i_pae":advanced_settings["weights_pae_inter"],
                                "con":advanced_settings["weights_con_intra"],
                                "i_con":advanced_settings["weights_con_inter"],
                                })

# redefine intramolecular contacts (con) and intermolecular contacts (i_con) definitions
af_model.opt["con"].update({"num":advanced_settings["intra_contact_number"],"cutoff":advanced_settings["intra_contact_distance"],"binary":False,"seqsep":9})
af_model.opt["i_con"].update({"num":advanced_settings["inter_contact_number"],"cutoff":advanced_settings["inter_contact_distance"],"binary":False})
    

### additional loss functions
if advanced_settings["use_rg_loss"]:
    # radius of gyration loss
    add_rg_loss(af_model, advanced_settings["weights_rg"])

if advanced_settings["use_i_ptm_loss"]:
    # interface pTM loss
    add_i_ptm_loss(af_model, advanced_settings["weights_iptm"])

if advanced_settings["use_termini_distance_loss"]:
    # termini distance loss
    add_termini_distance_loss(af_model, advanced_settings["weights_termini_loss"])

# add the helicity loss
add_helix_loss(af_model, helicity_value)

# calculate the number of mutations to do based on the length of the protein
greedy_tries = math.ceil(length * (advanced_settings["greedy_percentage"] / 100))

In [38]:
#### 4 stage ####
print("Stage 1: Test Logits")
af_model.design_logits(iters=50, e_soft=0.9, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True)


Stage 1: Test Logits


KeyboardInterrupt: 

In [2]:
!nvidia-smi

Fri Feb 21 10:20:45 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.15              Driver Version: 570.86.15      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:07.0 Off |                    0 |
| N/A   28C    P0             24W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----